In [1]:
import getpass
import os


def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Please provide your {var}")


_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("TAVILY_API_KEY")

Please provide your OPENAI_API_KEY ········
Please provide your TAVILY_API_KEY ········


In [2]:
! pip install langchain_experimental langsmith

In [2]:
from typing import Annotated

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools import PythonAstREPLTool

tavily_tool=TavilySearchResults(max_results=5)

python_repl_tool=PythonAstREPLTool()

D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\Narisetti Chand\AppData\Local\Temp\ipykernel_33784\574978722.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool=TavilySearchResults(max_results=5)


In [3]:
from langchain_core.messages import HumanMessage

def agent_node(state, agent, name):
    result=agent.invoke(state)
    return{
        "messages": [HumanMessage(content=result["messages"][-1].content, name=name)]
    }

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal

members = ["Researcher", "Coder"]
system_prompt=(
    "you are a supervisor tasked with managing a conversation between the"
    "following workers: {members}. given the following user request,"
    "respond wiht the worker to act next. each workder will perform a "
    "task and respond with their results and status. when finished "
    "respond with FINISH"
)
options = ["FINISH"] + members

class routeResponse(BaseModel):
    next: Literal[*options]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "Given the conversation above, who should act next?"
            " Or should we FINISH? Select one of: {options}",
        ),
    ]
).partial(options=str(options), members=",".join(members))

llm=ChatOpenAI(model="gpt-4o")

def supervisor_agent(state):
    supervisor_chain=prompt | llm.with_structured_output(routeResponse)
    return supervisor_chain.invoke(state)

In [5]:
import functools
import operator
from typing import Sequence
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next: str

research_agent = create_react_agent(llm, tools=[tavily_tool])
research_node = functools.partial(agent_node, agent=research_agent, name="Researcher")

code_agent = create_react_agent(llm, tools=[PythonAstREPLTool])
code_node = functools.partial(agent_node, agent=code_agent, name="Coder")

workflow=StateGraph(AgentState)
workflow.add_node("Researcher", research_node)
workflow.add_node("Coder", code_node)
workflow.add_node("supervisor", supervisor_agent)

C:\Users\Narisetti Chand\AppData\Local\Temp\ipykernel_33784\4172012322.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(llm, tools=[tavily_tool])
C:\Users\Narisetti Chand\AppData\Local\Temp\ipykernel_33784\4172012322.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  code_agent = create_react_agent(llm, tools=[PythonAstREPLTool])
D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'langchain_experimental.tools.python.tool.PythonInputs'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.w

In [6]:
for member in members:
    workflow.add_edge(member, "supervisor")

conditional_map = {k: k for k in members}
conditional_map["FINISH"] = END
workflow.add_conditional_edges("supervisor", lambda x: x["next"], conditional_map)

workflow.add_edge(START, "supervisor")

graph = workflow.compile()

In [7]:
for s in graph.stream(
    {
        "messages": [
            HumanMessage(content="Code hello world and print it to the terminal")
        ]
    }
):
    if "__end__" not in s:
        print(s)
        print("----")

{'supervisor': {'next': 'Coder'}}
----


D:\Download_New\anaconda\envs\ai_env\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `dict[any, any]` - serialized value may not be as expected [field_name='globals', input_value=<factory>, input_type=_HAS_DEFAULT_FACTORY_CLASS])
  PydanticSerializationUnexpectedValue(Expected `dict[any, any]` - serialized value may not be as expected [field_name='locals', input_value=<factory>, input_type=_HAS_DEFAULT_FACTORY_CLASS])
  return self.__pydantic_serializer__.to_python(


{'Coder': {'messages': [HumanMessage(content='It seems there was an error with executing the code. Let\'s demonstrate how you can write a simple "Hello, World!" program in Python:\n\n```python\n# Python code to print "Hello, World!" to the terminal\nprint("Hello, World!")\n```\n\nTo run this code, you can use any Python interpreter or a terminal with Python installed. Simply copy the script into a `.py` file and run it using `python filename.py`, or directly execute it in an interactive Python session.', additional_kwargs={}, response_metadata={}, name='Coder')]}}
----
{'supervisor': {'next': 'FINISH'}}
----


In [8]:
for s in graph.stream(
    {"messages": [HumanMessage(content="Write a report on langgraph.")]},
    {"recursion_limit": 100},
):
    if "__end__" not in s:
        print(s)
        print("----")

{'supervisor': {'next': 'Researcher'}}
----
{'Researcher': {'messages': [HumanMessage(content="LangGraph is a framework designed to build, manage, and deploy long-running, stateful AI agents. It focuses on agent orchestration through a visual and modular approach, allowing users to create stateful, multi-step workflows modeled as directed graphs. Here's a detailed report based on available data:\n\n### Overview\n\nLangGraph integrates seamlessly with LangChain, leveraging its components while providing additional capabilities for more complex workflows. It emphasizes the use of durable execution, streaming, and includes human-in-the-loop functionality. This makes it particularly suitable for developing AI systems where control and reliability are crucial.\n\n### Key Features\n\n1. **Agent Orchestration**: LangGraph excels in orchestrating AI agents by managing the flow and state across multiple nodes and edges in a graph structure.\n   \n2. **Node and State Management**: Agents are cha